In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "utils").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Project root with src/utils was not found")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from utils import (
    BereaPatchDataset,
    DigitalCorePipeline,
    FiLMRoutedUNet3D,
    PoreNetworkPermeabilityModel,
    check_required_dependencies,
)

check_required_dependencies()
device = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "device:", device)


ROOT: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct device: cuda


In [2]:
SEG_CKPT = ROOT / "models" / "film_routed_unet3d_best.pth"
GNN_CKPT = ROOT / "models" / "gnn_pnm_best.pth"

seg_checkpoint = torch.load(SEG_CKPT, map_location=device)
seg_model = FiLMRoutedUNet3D(
    base_channels=seg_checkpoint.get("base_channels", 16),
    ctx_dim=seg_checkpoint.get("ctx_dim", 64),
).to(device)
seg_model.load_state_dict(seg_checkpoint["model_state_dict"])

gnn_checkpoint = torch.load(GNN_CKPT, map_location=device)
graph_model = PoreNetworkPermeabilityModel(
    node_in=gnn_checkpoint["node_in"],
    edge_in=gnn_checkpoint["edge_in"],
    hidden=gnn_checkpoint.get("hidden", 64),
    layers=gnn_checkpoint.get("layers", 3),
    mu=1.0e-3,
).to(device)
graph_model.load_state_dict(gnn_checkpoint["model_state_dict"])
print("loaded checkpoints")


loaded checkpoints


In [3]:
CUBE_SIZE = 128
VOXEL_SIZE_M = 2.25e-6
DOMAIN_SIZE = (CUBE_SIZE * VOXEL_SIZE_M, CUBE_SIZE * VOXEL_SIZE_M, CUBE_SIZE * VOXEL_SIZE_M)

dataset = BereaPatchDataset(ROOT, split="val", cube_size=CUBE_SIZE, use_raw_gray=False, noise_types=["none"])
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
batch = next(iter(loader))
cube = batch["x"][0].numpy()
coord = batch["coord"][0].tolist()
print("coord:", coord, "cube:", cube.shape)


coord: [0, 0, 512] cube: (1, 128, 128, 128)


In [4]:
pipeline = DigitalCorePipeline(
    segmentation_model=seg_model,
    graph_model=graph_model,
    device=device,
    threshold=0.5,
    voxel_size=VOXEL_SIZE_M,
    mu=1.0e-3,
)

result = pipeline.run_cube(
    cube,
    input_is_pore_mask=False,
    domain_size=DOMAIN_SIZE,
    include_ph=True,
    compute_openpnm_baseline=True,
)

k_gnn = result.permeability.k_gnn_pnm.detach().cpu().numpy()
print("network:", result.network.metadata)
print("GNN+PNM k:", k_gnn)
print("OpenPNM k:", result.permeability.k_openpnm)


network: {'num_pores': 342, 'num_throats': 463, 'node_feature_dim': 12, 'edge_feature_dim': 12, 'ph_summary': [341.0, 0.007987558841705322, 6.352672062348574e-05, 120.0, 0.0008249112870544195, 6.168020627228543e-05]}
GNN+PNM k: [4.3094732e-14 9.4821855e-14 1.2960064e-13]
OpenPNM k: {'kx': 8.060985898077726e-14, 'ky': 6.337441314504612e-14, 'kz': 2.588886811886807e-13}


In [5]:
row = {
    "z": coord[0],
    "y": coord[1],
    "x": coord[2],
    "num_pores": result.network.metadata["num_pores"],
    "num_throats": result.network.metadata["num_throats"],
    "gnn_pnm_kx": float(k_gnn[0]),
    "gnn_pnm_ky": float(k_gnn[1]),
    "gnn_pnm_kz": float(k_gnn[2]),
}
if result.permeability.k_openpnm is not None:
    row.update({f"openpnm_{k}": v for k, v in result.permeability.k_openpnm.items()})

df = pd.DataFrame([row])
path = OUT_DIR / "full_pipeline_summary.csv"
df.to_csv(path, index=False)
print("saved:", path)
df


saved: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\full_pipeline_summary.csv


,z,y,x,num_pores,num_throats,gnn_pnm_kx,gnn_pnm_ky,gnn_pnm_kz,openpnm_kx,openpnm_ky,openpnm_kz
0,0,0,512,342,463,4.309473e-14,9.482186e-14,1.296006e-13,8.060986e-14,6.337441e-14,2.588887e-13
